# Store Sales — Time Series Forecasting
## Giải pháp 5-way Minimum-Variance Ensemble

**Bộ cạnh tranh:** [Kaggle Store Sales – Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)  
**Mục tiêu:** Dự báo doanh số 16 ngày cho 1782 chuỗi (54 cửa hàng × 33 nhóm hàng) tại Ecuador.  
**Metric:** RMSLE (Root Mean Squared Log Error) — thấp hơn = tốt hơn.

---

## Kiến trúc tổng thể

```
Dữ liệu thô (train.csv, test.csv, oil.csv, holidays_events.csv, stores.csv, transactions.csv)
        │
        ├── [LEG 1] Darts Family (6 biến thể cây GBDT)  ──────────── σ ≈ 0.379
        │     ├── base       : LightGBM depth-6, lags 56/7/365/730
        │     ├── deeper     : LightGBM depth-8 (300 vòng, lr=0.04)
        │     ├── xgb        : XGBoost + LightGBM (200 vòng, lr=0.05)
        │     ├── subsampled : 3-seed bagging
        │     ├── weighted   : sample_weight theo căn bậc hai
        │     └── cat_deep   : CatBoost depth-8 (200 vòng, lr=0.05)
        │
        ├── [LEG 2] LGBM-v8 (Direct Multi-step)  ─────────────────── σ ≈ 0.477
        │     └── 16 mô hình riêng (h=1..16), features phong phú + oil/holiday FE
        │
        ├── [LEG 3] Chronos-2 Covariates (Foundation LLM)  ────────── σ ≈ 0.398
        │     └── Amazon Chronos-2, conditioned on onpromotion + oil + holiday
        │
        ├── [LEG 4] TSMixer Tuned (Neural MLP-Mixer)  ─────────────── σ ≈ 0.382
        │     └── Global darts TSMixer, 1782 chuỗi, HID=128, BLK=8, 30 epochs
        │
        └── [LEG 5] TiDE (Neural, future+static covariates)  ──────── σ ≈ 0.391
              └── Global darts TiDE, fast profile (icl=60, 12 epochs)
                        │
                        ▼
        5-way Minimum-Variance Ensemble (log1p space)
        weights ≈ [0.477, 0.007, 0.175, 0.276, 0.066] (fam/v8/cov/tsm/tide)
                        │
                        ▼
        submission_fam_cov_v8_tsm_tide_5way.csv  →  RMSLE ≈ 0.37379
```

---
## Chương 1 — Cài đặt & Dữ liệu

### 1.1 Môi trường và đường dẫn

In [1]:
import os, sys, time, subprocess, shlex
from pathlib import Path
import pandas as pd
import numpy as np

def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config.yaml").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Không tìm thấy thư mục gốc repo!")

REPO = _find_root(Path.cwd())
SRC  = REPO / "src"
DATA = REPO / "data"
SUBMISSIONS = REPO / "submissions"
SUBMISSIONS.mkdir(exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from store_sales.config import get_config
from store_sales.ensemble import blend

cfg = get_config()
print(f"REPO        = {REPO}")
print(f"SRC         = {SRC}")
print(f"DATA        = {DATA}")
print(f"SUBMISSIONS = {SUBMISSIONS}")

REPO        = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting
SRC         = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/src
DATA        = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/data
SUBMISSIONS = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/submissions


### 1.2 Kiểm tra dữ liệu đầu vào

In [2]:
expected = ["train.csv", "test.csv", "stores.csv", "oil.csv",
            "holidays_events.csv", "transactions.csv"]
present  = [f for f in expected if (DATA / f).exists()]
missing  = [f for f in expected if f not in present]
print(f"Tìm thấy: {present}")
if missing:
    raise FileNotFoundError(f"Thiếu file: {missing} — đặt vào thư mục data/")
else:
    print("Tất cả file dữ liệu đã sẵn sàng.")

Tìm thấy: ['train.csv', 'test.csv', 'stores.csv', 'oil.csv', 'holidays_events.csv', 'transactions.csv']
Tất cả file dữ liệu đã sẵn sàng.


### 1.3 Công tắc chạy

| Biến | Giá trị | Ý nghĩa |
|---|---|---|
| `RUN_LEGS` | `True` | Huấn luyện lại mô hình nếu file CSV chưa tồn tại |
| `RUN_LEGS` | `False` | Chỉ chạy bước Blend từ các file CSV đã có |
| `FORCE_RETRAIN` | `True` | Bắt buộc train lại dù file CSV đã tồn tại |

> **Khuyến nghị:** `RUN_LEGS=True, FORCE_RETRAIN=False` để chỉ train những mô hình chưa có file.

In [3]:
RUN_LEGS     = True   # True = train legs nếu chưa có CSV; False = blend-only
FORCE_RETRAIN = True  # True = train lại dù CSV đã tồn tại
TIMINGS = {}           # dict ghi thời gian train từng mô hình

def run(cmd: str, produces: str = "", *, force: bool = FORCE_RETRAIN) -> None:
    label = produces or cmd.split()[0]
    out_path = SUBMISSIONS / produces if produces else None

    if out_path and out_path.exists() and not force:
        print(f"  [exists] {produces}  (bỏ qua)")
        return

    if not RUN_LEGS:
        print(f"  [skip]   {label}  (RUN_LEGS=False)")
        return

    args = shlex.split(cmd)

    if args and args[0] == "python":
        args[0] = sys.executable

    env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}
    print(f"  [run]    {label}")
    t0 = time.time()
    subprocess.run(args, env=env, cwd=str(REPO), check=True)
    elapsed = time.time() - t0
    TIMINGS[label] = elapsed
    print(f"  [done]   {label}  ({elapsed/60:.1f} min)")

print(f"RUN_LEGS={RUN_LEGS}, FORCE_RETRAIN={FORCE_RETRAIN}")

RUN_LEGS=True, FORCE_RETRAIN=True


### 1.4 Cài thư viện (nếu cần)

In [4]:
if RUN_LEGS:
    print("Kiểm tra thư viện...")
    try:
        import lightgbm, xgboost, catboost, darts
        print("lightgbm, xgboost, catboost, darts — đã cài sẵn")
    except ImportError as e:
        print(f"Thiếu thư viện: {e}. Đang cài...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "lightgbm", "xgboost", "catboost", "darts"], check=True)
        print("Cài xong")
else:
    print("RUN_LEGS=False — bỏ qua kiểm tra thư viện.")

Kiểm tra thư viện...
lightgbm, xgboost, catboost, darts — đã cài sẵn


---
## Chương 2 — Huấn luyện các mô hình

> Mỗi cell dưới đây train một nhánh (leg). Cell chỉ thực sự chạy khi:
> - `RUN_LEGS = True`, **VÀ**
> - File CSV đầu ra chưa tồn tại (hoặc `FORCE_RETRAIN = True`)

### 2.1 LEG 1 — Darts Family (6 biến thể cây GBDT)

Cùng một khung Darts nhưng khác nhau về thuật toán, độ sâu cây, và chiến lược lấy mẫu.  
Mỗi biến thể huấn luyện **4 mô hình lồng nhau** (lags 56/7/365/730 ngày) rồi lấy trung bình.

| Biến thể | Thuật toán | Đặc điểm | File output |
|---|---|---|---|
| `base` | LightGBM | depth-6 + FE (oil+holiday) | `submission_darts_lgbm.csv` |
| `deeper` | LightGBM | depth=8, **n_est=300, lr=0.04** | `submission_darts_lgbm_deeper.csv` |
| `xgb` | XGBoost+LGBM | + XGBoost 4 configs, **n_est=200, lr=0.05** | `submission_darts_xgb_lgbm.csv` |
| `subsampled` | LightGBM | Trung bình 3 seeds | `submission_darts_lgbm_subsampled_3seed.csv` |
| `weighted` | LightGBM | Đánh trọng số √ theo thời gian | `submission_darts_lgbm_w.csv` |
| `cat_deep` | CatBoost | depth=8, **n_est=200, lr=0.05** | `submission_darts_cat_deeper.csv` |

> **Lưu ý:** `n_estimators` 100→200/300 cho `deeper`/`xgb`/`cat_deep` là đòn bẩy chính kéo
> family sub-blend từ ~0.382 xuống **0.37854** (các cây sâu hơn cần nhiều vòng boosting hơn).

In [ ]:
run("python -m store_sales.cli train darts-family --variant base",
    produces="submission_darts_lgbm.csv")

In [ ]:
run("python -m store_sales.cli train darts-family --variant deeper",
    produces="submission_darts_lgbm_deeper.csv")

In [ ]:
run("python -m store_sales.cli train darts-family --variant xgb",
    produces="submission_darts_xgb_lgbm.csv")

In [ ]:
run("python -m store_sales.cli train darts-family --variant subsampled",
    produces="submission_darts_lgbm_subsampled_3seed.csv")

In [ ]:
run("python -m store_sales.cli train darts-family --variant weighted",
    produces="submission_darts_lgbm_w.csv")

In [ ]:
run("python -m store_sales.cli train darts-family --variant cat_deep",
    produces="submission_darts_cat_deeper.csv")

### 2.2 LEG 2 — LGBM-v8 (Direct Multi-step Forecasting)

Thay vì dự đoán toàn bộ 16 ngày cùng một lúc, v8 huấn luyện **16 mô hình LightGBM độc lập**
(một mô hình cho mỗi ngày trong horizon h=1..16).  
Điều này giúp mỗi mô hình học được pattern riêng của từng ngày dự báo.

**Features đặc biệt của v8:**
- Lag ngắn (1-21 ngày) + lag dài (28, 35, ..., 364 ngày)  
- Rolling windows (7, 14, 28, 56, 84, 168 ngày)  
- EWM half-life (7, 28 ngày)  
- Oil dynamics (biến động giá dầu)  
- Holiday distance (khoảng cách đến ngày lễ gần nhất)  
- Promotional features (tương tác khuyến mãi × ngày trong tuần)

In [11]:
run("python -m store_sales.cli train lgbm-v8 --suffix reg",
    produces="submission_v8_reg.csv")

  [run]    submission_v8_reg.csv
Cutoffs: train_from=2015-01-01 val=2017-07-31 test=[2017-08-16..2017-08-31]
Objective=regression seeds=[42] horizons=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Always-zero pairs (pre-2017-08-16): 53
Rows before zero-pair drop: 3,029,400; after: 2,939,300
  add_sf_lag_features...
  add_sf_rolling_per_h...
  add_dow_baseline_per_h...
  add_aggregate_lag_roll...
  add_promo_features...
  add_days_since_first_sale...
Total feature pool: 206
Train rows: 1,625,260  Val rows: 27,664  Test rows: 27,664

=== Probe (h=16, ES, seed=42, obj=regression) ===
Training until validation scores don't improve for 150 rounds
[400]	val's rmse: 0.41739
[800]	val's rmse: 0.413856
[1200]	val's rmse: 0.412404
Early stopping, best iteration is:
[1373]	val's rmse: 0.411732
Probe best_iter=1373, fixed_iter=1441

=== Per-horizon (16 horizons, 1 seeds) ===

-- h=1: 140 features, h_base=1
   h=1 val RMSLE: 0.38824  elapsed=281s

-- h=2: 133 features, h_base=4
   h=2 val 

### 2.3 LEG 3 — Chronos-2 với Covariates (Foundation LLM)

Mô hình ngôn ngữ lớn dành cho chuỗi thời gian của Amazon, được điều kiện hoá trên:
- **`onpromotion`**: biến động khuyến mãi (known-future covariate)  
- **`oil`**: giá dầu WTI (past-only covariate)  
- **`holiday`**: cờ ngày lễ quốc gia (known-future covariate)

> **Yêu cầu đặc biệt:** Chronos-2 cần **virtual environment riêng** với Python 3.11  
> và `chronos-forecasting==2.2.2` vì có conflict dependency với các thư viện khác.
>
> **Thời gian chạy:** ~15-30 phút (CPU inference)  
> **Lần đầu:** Tự động tải model ~2GB từ HuggingFace về cache.

**Thiết lập venv:**
```bash
python3.11 -m venv .venv_chronos2
.venv_chronos2/bin/pip install "chronos-forecasting==2.2.2" numpy pandas pyarrow
```

In [12]:
# Kiểm tra .venv_chronos2 đã tồn tại chưa
venv_python = REPO / ".venv_chronos2/bin/python"
if not venv_python.exists():
    print("   Chưa có .venv_chronos2!")
    print("   Chạy lệnh sau để tạo:")
    print("   python3.11 -m venv .venv_chronos2")
    print("   .venv_chronos2/bin/pip install 'chronos-forecasting==2.2.2' numpy pandas pyarrow")
else:
    import subprocess as _sp
    r = _sp.run([str(venv_python), "-c", "from chronos import Chronos2Pipeline; print('OK')"],
                capture_output=True, text=True, cwd=str(REPO))
    if r.returncode == 0:
        print(".venv_chronos2 sẵn sàng — Chronos2Pipeline import OK")
    else:
        print("Lỗi import Chronos2:", r.stderr[:200])

.venv_chronos2 sẵn sàng — Chronos2Pipeline import OK


In [13]:
# Train Chronos-2 với oil + holiday covariates
# Tạo ra: submissions/submission_chronos2_cov_promo_oil_hol.csv
run(f"{venv_python} -m store_sales.cli train chronos2-cov --oil --holiday --suffix _promo_oil_hol",
    produces="submission_chronos2_cov_promo_oil_hol.csv")

  [run]    submission_chronos2_cov_promo_oil_hol.csv
Loading amazon/chronos-2 ... (oil=True holiday=True)
OK

[TEST] window 2017-08-16..08-31
  panel built (1782 series) in 7s; predicting...


/Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/.venv_chronos2/lib/python3.11/site-packages/chronos/chronos2/dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  task_target = torch.from_numpy(task_target)


  done in 68s
Saved submission_chronos2_cov_promo_oil_hol.csv: rows=28512 mean=449.61
  [done]   submission_chronos2_cov_promo_oil_hol.csv  (1.3 min)


### 2.4 LEG 4 — TSMixer (Global Neural Forecaster)

Mô hình mạng nơ-ron MLP-Mixer huấn luyện **toàn cục** trên tất cả 1782 chuỗi cùng một lúc.  
Covariates: khuyến mãi, giá dầu, lịch, ngày lễ.

**Cấu hình tuned:** `HID=128, FF=128, BLK=8, DROP=0.2, LR=1e-3, epochs=30`  
**Scheduler:** CosineAnnealingLR

In [14]:
# TSMixer: dùng file đã train sẵn nếu tồn tại (tiết kiệm 6+ tiếng)
# run("python -m store_sales.cli train tsmixer --epochs 30 --cpu --mps",
#     produces="submission_tsmixer_tuned.csv")

### 2.5 LEG 5 — TiDE (neural thứ 2)

Thêm một leg neural **kiến trúc khác TSMixer** để tăng đa dạng cho blend: lỗi ít tương quan →
min-var tự gán trọng số và giảm thêm RMSLE. TiDE dùng known-future + static covariates
(σ≈0.391, weight ~0.07 trong blend).

Cell train có công tắc riêng (`EXTRA_NEURAL`, `NEURAL_EPOCHS`, ...), độc lập với `RUN_LEGS`.
Cell "Xem trước" ước lượng `math_LB` khi thêm một leg mới — kiểm tra trước khi tốn lượt nộp Kaggle.

In [15]:
# Train leg neural bổ sung (TiDE/NHiTS) — độc lập với RUN_LEGS
EXTRA_NEURAL  = ["tide"]        # ["tide"] | ["nhits"] | ["tide", "nhits"]
NEURAL_EPOCHS = 12              # fast profile (mặc định cũ 30)
NEURAL_ICL    = 60              # input_chunk_length (config=90)
NEURAL_BATCH  = 2048            # batch size (config=1024)
NEURAL_DEVICE = "--cpu --mps"   # Mac: "--cpu --mps" | CUDA: "--gpu" | CPU: "--cpu"
FORCE_NEURAL  = False           # True = train lại dù file đã có

_env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}
for _m in EXTRA_NEURAL:
    _out = f"submission_{_m}.csv"
    if (SUBMISSIONS / _out).exists() and not FORCE_NEURAL:
        print(f"  [exists] {_out}")
        continue
    _cmd = (f"{sys.executable} -m store_sales.cli train neural "
            f"--model {_m} --epochs {NEURAL_EPOCHS} --icl {NEURAL_ICL} "
            f"--batch-size {NEURAL_BATCH} {NEURAL_DEVICE}")
    print(f"  [run]    {_out}")
    _t0 = time.time()
    subprocess.run(shlex.split(_cmd), env=_env, cwd=str(REPO), check=True)
    print(f"  [done]   {_out}  ({(time.time() - _t0)/60:.1f} phút)")

  [exists] submission_tide.csv


In [16]:
# Xem trước math_LB khi thêm một leg mới (chưa cần sửa config)
from store_sales.io.submissions import load_log

NEW_LEG_FILE  = "submission_<ten_leg_moi>.csv"   # file leg mới (chưa có trong config)
NEW_LEG_SIGMA = 0.400                             # σ đo trên Kaggle (hoặc ước lượng để thử)

get_config.cache_clear()
ens = get_config().ensemble
fam_log, _ = blend.build_family(ens.family_alpha)
cur  = list(ens.leg_sigma)
logs = [fam_log if l == "family" else load_log(ens.leg_files[l]) for l in cur]
sig  = [ens.leg_sigma[l] for l in cur]
_, lb_cur, _, _ = blend.min_var_weights(sig, logs)

new_name = NEW_LEG_FILE.replace("submission_", "").replace(".csv", "")
if new_name in ens.leg_files:
    print(f"'{new_name}' đã có trong config. math_LB hiện tại = {lb_cur:.5f}")
elif not (SUBMISSIONS / NEW_LEG_FILE).exists():
    print(f"Chưa thấy {NEW_LEG_FILE}. math_LB hiện tại = {lb_cur:.5f}")
else:
    w_new, lb_new, _, _ = blend.min_var_weights(sig + [NEW_LEG_SIGMA], logs + [load_log(NEW_LEG_FILE)])
    display(pd.DataFrame({"leg": cur + [new_name],
                          "sigma":  [f"{s:.5f}" for s in sig + [NEW_LEG_SIGMA]],
                          "weight": [f"{w:.4f}" for w in w_new]}))
    print(f"math_LB hiện tại = {lb_cur:.5f}  →  thêm {new_name} = {lb_new:.5f}  (delta {lb_new - lb_cur:+.5f})")
    print("delta < 0 = leg mới giúp blend → thêm vào config rồi chạy §3.3.")

Chưa thấy submission_<ten_leg_moi>.csv. math_LB hiện tại = 0.37379


---
## Chương 3 — Ensemble & Kết quả

### 3.1 Kiểm tra đầy đủ các file submission

Trước khi trộn, xác nhận tất cả file CSV của các nhánh đã tồn tại trong `submissions/`.

In [17]:
get_config.cache_clear()
cfg = get_config()

needed = (
    list(cfg.ensemble.family_files.values()) +
    list(cfg.ensemble.leg_files.values())
)

all_ok = True
for fname in needed:
    path = SUBMISSIONS / fname
    exists = path.exists()
    size_kb = f"{path.stat().st_size//1024} KB" if exists else "---"
    status = "success" if exists else "fail"
    print(f"  {status}  {fname:<52} {size_kb}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nTất cả file sẵn sàng — có thể chạy bước blend!")
else:
    print("\nCòn thiếu file — hãy train lại các mô hình bị thiếu ở Chương 2.")

  success  submission_darts_lgbm.csv                            708 KB
  success  submission_darts_lgbm_deeper.csv                     708 KB
  success  submission_darts_xgb_lgbm.csv                        708 KB
  success  submission_darts_lgbm_subsampled_3seed.csv           708 KB
  success  submission_darts_cat_deeper.csv                      709 KB
  success  submission_darts_lgbm_w.csv                          708 KB
  success  submission_chronos2_cov_promo_oil_hol.csv            497 KB
  success  submission_v8_reg.csv                                722 KB
  success  submission_tsmixer_tuned.csv                         716 KB
  success  submission_tide.csv                                  721 KB

Tất cả file sẵn sàng — có thể chạy bước blend!


### 3.2 Trọng số blend & RMSLE ước lượng

Hàm `build_fourway()` tính **minimum-variance weights** dựa trên ma trận hiệp phương sai  
được ước lượng từ các điểm số σ_i (RMSLE trên Kaggle Leaderboard) và mức độ khác biệt  
giữa các dự đoán của từng nhánh.

**Công thức:** `Cov_ij = (σ_i² + σ_j² − D_ij²) / 2`  
trong đó `D_ij = RMS(log_pred_i − log_pred_j)` — hai nhánh khác biệt nhiều → bổ sung tốt hơn.

In [18]:
get_config.cache_clear()
cfg = get_config()
ens = cfg.ensemble

# Bảng sigma từng nhánh
print("=== Sigma (RMSLE standalone trên Kaggle LB) ===\n")
rows = []
for m in ens.family_files:
    rows.append({"leg": f"family.{m}", "sigma": ens.family_sigma[m],
                 "file": ens.family_files[m]})
for leg in ens.leg_files:
    rows.append({"leg": leg, "sigma": ens.leg_sigma[leg],
                 "file": ens.leg_files[leg]})
df_sigma = pd.DataFrame(rows).sort_values("sigma")
display(df_sigma.style.format({"sigma": "{:.5f}"}))

# Tính trọng số blend (số leg theo config — hiện 5-way)
print("\n=== Blend Weights & Estimated RMSLE ===\n")
r = blend.build_fourway()
table = pd.DataFrame([{
    "leg": leg,
    "sigma": ens.leg_sigma.get(leg, ens.family_sigma.get(leg, "-")),
    "weight": f"{w:.4f}",
} for leg, w in zip(r["legs"], r["weights"])])
display(table)
print(f"\nEstimated math_LB = {r['blend_rmsle_formula']:.5f}")
print("(Điểm thực tế trên Kaggle thường chênh ±0.0002 so với ước lượng này)")

=== Sigma (RMSLE standalone trên Kaggle LB) ===



,leg,sigma,file
2,family.xgb,0.38036,submission_darts_xgb_lgbm.csv
1,family.deeper,0.38093,submission_darts_lgbm_deeper.csv
8,tsmixer,0.38191,submission_tsmixer_tuned.csv
4,family.cat_deep,0.38248,submission_darts_cat_deeper.csv
3,family.sub3,0.38298,submission_darts_lgbm_subsampled_3seed.csv
0,family.base,0.38337,submission_darts_lgbm.csv
5,family.weighted,0.38604,submission_darts_lgbm_w.csv
9,tide,0.39136,submission_tide.csv
6,chronos2-cov,0.39772,submission_chronos2_cov_promo_oil_hol.csv
7,lgbm-reg,0.47682,submission_v8_reg.csv



=== Blend Weights & Estimated RMSLE ===



,leg,sigma,weight
0,family,0.37853,0.4767
1,chronos2-cov,0.39772,0.1745
2,lgbm-reg,0.47682,0.0066
3,tsmixer,0.38191,0.2764
4,tide,0.39136,0.0658



Estimated math_LB = 0.37379
(Điểm thực tế trên Kaggle thường chênh ±0.0002 so với ước lượng này)


### 3.3 Tạo file nộp cuối cùng

Cell này chạy `store_sales.cli blend build` để:
1. Tính minimum-variance weights
2. Trộn các nhánh (hiện 5-way) trong log1p-space
3. Ghi ra file nộp cuối cùng

In [19]:
env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}

print("=== Bước 1: Build final submission ===")
build_result = subprocess.run(
    [sys.executable, "-m", "store_sales.cli", "blend", "build"],
    env=env, cwd=str(REPO), capture_output=False, text=True
)

print("\n=== Bước 2: Verify byte-exact reproduction ===")
verify_result = subprocess.run(
    [sys.executable, "-m", "store_sales.cli", "blend", "verify"],
    env=env, cwd=str(REPO), capture_output=False, text=True
)

# Kiểm tra file đầu ra
get_config.cache_clear()
cfg = get_config()
out_file = SUBMISSIONS / cfg.ensemble.out_file
if out_file.exists():
    sub = pd.read_csv(out_file)
    print(f"\nFile nộp đã tạo: {out_file.name}")
    print(f"   Rows: {len(sub):,} | Mean sales: {sub['sales'].mean():.2f} | Max: {sub['sales'].max():.0f}")
else:
    print("Không tìm thấy file nộp!")

=== Bước 1: Build final submission ===
weights family=0.4767 chronos2-cov=0.1745 lgbm-reg=0.0066 tsmixer=0.2764 tide=0.0658
math_LB = 0.37379  (formula estimate)
diff family<->tsmixer = 0.123 (family-level redundancy watch)
Wrote submission_fam_cov_v8_tsm_tide_5way.csv  (rows=28512, max sales=14396)
Wrote submission_family.csv  (rows=28512)

=== Bước 2: Verify byte-exact reproduction ===
weights family=0.4767 chronos2-cov=0.1745 lgbm-reg=0.0066 tsmixer=0.2764 tide=0.0658
math_LB = 0.37379  (formula estimate)
diff family<->tsmixer = 0.123 (family-level redundancy watch)
VERIFY max|delta| = 1.819e-12  ->  EXACT MATCH

File nộp đã tạo: submission_fam_cov_v8_tsm_tide_5way.csv
   Rows: 28,512 | Mean sales: 436.81 | Max: 14396


### 3.4 Thời gian huấn luyện từng mô hình

In [20]:
if TIMINGS:
    items = sorted(TIMINGS.items(), key=lambda kv: kv[1], reverse=True)
    df_time = pd.DataFrame([
        {"model": k, "minutes": round(v/60, 1), "seconds": round(v, 0)}
        for k, v in items
    ])
    display(df_time)
    print(f"\nTổng thời gian train: {sum(TIMINGS.values())/60:.1f} phút")
else:
    print("Không có mô hình nào được train trong phiên này (tất cả đã dùng file sẵn có).")

,model,minutes,seconds
0,submission_darts_xgb_lgbm.csv,77.9,4676.0
1,submission_darts_cat_deeper.csv,62.4,3744.0
2,submission_v8_reg.csv,58.0,3481.0
3,submission_darts_lgbm_deeper.csv,28.1,1685.0
4,submission_darts_lgbm_subsampled_3seed.csv,14.7,879.0
5,submission_darts_lgbm.csv,13.3,798.0
6,submission_darts_lgbm_w.csv,12.3,740.0
7,submission_chronos2_cov_promo_oil_hol.csv,1.3,80.0



Tổng thời gian train: 268.1 phút


---
## Chương 4 — Hướng dẫn nộp bài

### 4.1 File nộp cuối cùng

| File | Mô tả | RMSLE ước tính |
|---|---|---|
| `submission_fam_cov_v8_tsm_tide_5way.csv` | **5-way Ensemble** (nộp bài chính) | **≤ 0.37379** |
| `submission_family.csv` | Family sub-blend (6 GBDT) | ≈ 0.37854 |
| `submission_darts_lgbm.csv` | LightGBM base | ≈ 0.38337 |
| `submission_darts_lgbm_deeper.csv` | LightGBM depth-8 (300 vòng) | ≈ 0.38093 |
| `submission_darts_xgb_lgbm.csv` | XGBoost+LightGBM (200 vòng) | ≈ 0.38036 |
| `submission_darts_cat_deeper.csv` | CatBoost depth-8 (200 vòng) | ≈ 0.38248 |
| `submission_tsmixer_tuned.csv` | TSMixer | ≈ 0.38191 |
| `submission_chronos2_cov_promo_oil_hol.csv` | Chronos-2 oil+hol | ≈ 0.39772 |
| `submission_tide.csv` | TiDE (neural) | ≈ 0.39136 |

### 4.2 Nộp file lên Kaggle

```bash
# Cài Kaggle CLI (nếu chưa có)
pip install kaggle

# Nộp file ensemble
kaggle competitions submit \
    -c store-sales-time-series-forecasting \
    -f submissions/submission_fam_cov_v8_tsm_tide_5way.csv \
    -m "5-way minvar ensemble: family+chronos2-oilhol+v8+tsmixer+tide"
```

### 4.3 Cập nhật sigma sau khi nộp

Sau khi nộp từng file lên Kaggle và nhận điểm RMSLE thực tế,  
cập nhật vào `config.yaml` phần `ensemble.family_sigma` / `ensemble.leg_sigma`,  
rồi chạy lại **Chương 3** để tính lại weights tối ưu.

### 4.4 Kết luận

| Tiêu chí | Kết quả |
|---|---|
| **Metric đạt được** | RMSLE = 0.37379 (public leaderboard) |
| **Phương pháp cốt lõi** | 5-way minimum-variance ensemble (covariance neg-weight) |
| **Mô hình quan trọng nhất** | Darts Family (weight ≈ 48%) + TSMixer (≈ 28%) |
| **Nguồn cải thiện chính** | σ hiệu chỉnh trung thực + TiDE + retune GBT (deeper n_est=300, xgb/cat_deep n_est=200) |

> **Insight:** Các mô hình "yếu" như Chronos-2 (σ=0.40) và LGBM-v8 (σ=0.48) lại có đóng góp lớn  
> vì **lỗi của chúng không tương quan** với các mô hình cây — giúp blend triệt tiêu được phần sai số chung.